# 04. Pseudo-Labeling Experiment

This notebook generates pseudo-labels from a trained teacher model and trains a model using a mixture of labeled and pseudo-labeled examples.
It supports the report section on pseudo-label generation, confidence thresholding, and soft labels.

In [ ]:
# Common setup
from pathlib import Path
import sys
import os
import json
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Find project root: works when notebook is run from repo root or from notebooks/
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src").exists():
    # Walk upward in case the notebook is launched from a subfolder
    for parent in Path.cwd().resolve().parents:
        if (parent / "src").exists():
            PROJECT_ROOT = parent
            break

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

DATA_DIR = PROJECT_ROOT / "data"
REPORT_DIR = PROJECT_ROOT / "reports"
FIG_DIR = REPORT_DIR / "figures"
TABLE_DIR = REPORT_DIR / "tables"
EXP_DIR = PROJECT_ROOT / "experiments" / "notebook_outputs"
for p in [FIG_DIR, TABLE_DIR, EXP_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Source path:", SRC_DIR)

import torch
from torch.utils.data import DataLoader, Dataset

from bioacoustic.dataset import read_metadata, build_class_list, add_stratified_folds, BirdAudioDataset, encode_multihot, make_label_map
from bioacoustic.audio import load_audio, make_regular_windows
from bioacoustic.spectrogram import batch_log_mel
from bioacoustic.models import build_model
from bioacoustic.losses import focal_bce_loss, soft_label_bce_loss, compute_pos_weight
from bioacoustic.training import train_one_epoch, validate, save_checkpoint, load_checkpoint
from bioacoustic.pseudo_labeling import select_pseudo_labels, save_pseudo_labels, load_pseudo_labels, mix_labeled_and_pseudo
from bioacoustic.utils import seed_everything, get_device

seed_everything(42)
DEVICE = get_device(True)
print("Device:", DEVICE)

## Configuration

In [ ]:
CFG = {
    "metadata_path": DATA_DIR / "train_metadata.csv",
    "audio_dir": DATA_DIR / "train_audio",
    "unlabeled_audio_dir": DATA_DIR / "unlabeled_soundscapes",
    "teacher_checkpoint": EXP_DIR / "sed" / "best_sed.pt",
    "output_dir": EXP_DIR / "pseudo_labeling",
    "sample_rate": 32000,
    "duration": 5.0,
    "n_fft": 2048,
    "hop_length": 768,
    "n_mels": 192,
    "f_min": 50,
    "f_max": 15000,
    "backbone": "tf_efficientnet_b0_ns",
    "batch_size": 12,
    "num_workers": 2,
    "epochs": 3,
    "lr": 1e-4,
    "fold": 0,
    "n_splits": 5,
    "max_unlabeled_files": 200,
    "min_max_prob": 0.5,
    "class_prob_threshold": 0.1,
    "labeled_ratio": 0.6,
}
CFG["output_dir"].mkdir(parents=True, exist_ok=True)

## Load teacher and generate pseudo-label predictions

In [ ]:
df = read_metadata(CFG["metadata_path"])
classes = build_class_list(df)
num_classes = len(classes)
spec_kwargs = dict(n_fft=CFG["n_fft"], hop_length=CFG["hop_length"], n_mels=CFG["n_mels"], f_min=CFG["f_min"], f_max=CFG["f_max"])

teacher = build_model("sed", num_classes=num_classes, backbone=CFG["backbone"], in_channels=1, pretrained=False).to(DEVICE)
if CFG["teacher_checkpoint"].exists():
    load_checkpoint(CFG["teacher_checkpoint"], teacher, map_location=DEVICE)
else:
    print("WARNING: teacher checkpoint not found. Generate one from the SED notebook first:", CFG["teacher_checkpoint"])
teacher.eval()

unlabeled_files = sorted(list(Path(CFG["unlabeled_audio_dir"]).glob("*.ogg")) + list(Path(CFG["unlabeled_audio_dir"]).glob("*.wav")))
if CFG["max_unlabeled_files"]:
    unlabeled_files = unlabeled_files[:CFG["max_unlabeled_files"]]
print("Unlabeled files:", len(unlabeled_files))

all_probs = []
row_ids = []
with torch.no_grad():
    for audio_path in unlabeled_files:
        wav = load_audio(audio_path, sample_rate=CFG["sample_rate"])
        windows = make_regular_windows(wav, sample_rate=CFG["sample_rate"], duration=CFG["duration"])
        x = batch_log_mel(windows, sample_rate=CFG["sample_rate"], **spec_kwargs)
        x = torch.tensor(x, dtype=torch.float32).to(DEVICE)
        out = teacher(x)
        probs = torch.sigmoid(out["clip_logits"]).detach().cpu().numpy()
        all_probs.append(probs)
        for i in range(len(probs)):
            row_ids.append(f"{audio_path.stem}_{(i+1)*int(CFG['duration'])}")

if all_probs:
    predictions = np.concatenate(all_probs, axis=0)
else:
    predictions = np.zeros((0, num_classes), dtype=np.float32)

print("Pseudo prediction matrix:", predictions.shape)
np.save(CFG["output_dir"] / "unlabeled_predictions.npy", predictions)

## Select pseudo labels

In [ ]:
pseudo_df = select_pseudo_labels(
    predictions,
    row_ids=row_ids,
    min_max_prob=CFG["min_max_prob"],
    class_prob_threshold=CFG["class_prob_threshold"],
)
print("Selected pseudo chunks:", len(pseudo_df), "/", len(predictions))
if len(predictions):
    print("Selection ratio:", len(pseudo_df) / len(predictions))

pseudo_path = CFG["output_dir"] / "pseudo_labels.csv"
save_pseudo_labels(pseudo_df, pseudo_path)
display(pseudo_df.head())

## Pseudo-label quality analysis

In [ ]:
if len(pseudo_df):
    prob_cols = [c for c in pseudo_df.columns if c != "row_id"]
    max_probs = pseudo_df[prob_cols].max(axis=1)
    nonzero_counts = (pseudo_df[prob_cols].values > 0).sum(axis=1)

    plt.figure(figsize=(7, 4))
    max_probs.hist(bins=30)
    plt.title("Selected pseudo-label max probability")
    plt.xlabel("Max probability")
    plt.ylabel("Chunks")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "pseudo_label_max_probability.png", dpi=200)
    plt.show()

    plt.figure(figsize=(7, 4))
    pd.Series(nonzero_counts).value_counts().sort_index().plot(kind="bar")
    plt.title("Number of retained classes per pseudo-labeled chunk")
    plt.xlabel("Retained classes")
    plt.ylabel("Chunks")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "pseudo_label_retained_classes.png", dpi=200)
    plt.show()

## Train with labeled + pseudo-labeled data

This cell demonstrates the training stage. For pseudo-labeled rows, a production implementation should use their saved soft-target vectors directly. The core idea is to combine labeled data and pseudo-labeled data using a controlled ratio, such as 60:40.

In [ ]:
# Keep this section as a reproducible template. In most projects, pseudo-labeled rows
# are converted into a dedicated dataset class that reads audio chunk paths and soft targets.
# The code below saves the mixed index table for the training script.
if "fold" not in df.columns:
    df = add_stratified_folds(df, n_splits=CFG["n_splits"], seed=42)
train_df = df[df["fold"] != CFG["fold"]].reset_index(drop=True)
mixed_df = mix_labeled_and_pseudo(train_df, pseudo_df, labeled_ratio=CFG["labeled_ratio"], seed=42)
mixed_path = CFG["output_dir"] / "mixed_labeled_pseudo_index.csv"
mixed_df.to_csv(mixed_path, index=False)
print("Mixed training table saved to:", mixed_path)
display(mixed_df.head())
print(mixed_df["source"].value_counts())